# C-MAPSS FD003 — Sequential CNN-LSTM optimized with GA and Hyperband (Option C)

This notebook optimizes the **sequential** CNN-LSTM on C-MAPSS FD003 using the same two
optimizers (Genetic Algorithm and Hyperband) that were applied to the **parallel** model,
so the two architectures are compared under equal optimization on this dataset.

Pipeline matches the reported FD003 experiments: condition-specific scaling, exponential
smoothing (alpha 0.4), RUL clipped at 125, 14 selected sensors, sliding window of 20,
engine-unit-grouped 80/20 train/validation split. Baseline sequential test RMSE = 15.31.

Run on a machine with the FD003 files and a GPU (an RTX 3090 was used).

In [1]:
%matplotlib inline
import os, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Masking, Conv1D, MaxPooling1D, Dropout,
                                      TimeDistributed, Flatten, LSTM, Dense)
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from deap import base, creator, tools, algorithms
import keras_tuner as kt

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TF', tf.__version__)

TF 2.10.0


In [2]:
# ---- data paths (relative to the env_eweda folder) ----
DATA_DIR   = 'CMaps/'
train_file = 'train_FD003.txt'
test_file  = 'test_FD003.txt'
rul_file   = 'RUL_FD003.txt'

index_names   = ['unit_nr', 'time_cycles']
setting_names = ['setting_1', 'setting_2', 'setting_3']
sensor_names  = ['s_{}'.format(i+1) for i in range(21)]
col_names     = index_names + setting_names + sensor_names

train  = pd.read_csv(DATA_DIR+train_file, sep=r'\s+', header=None, names=col_names)
test   = pd.read_csv(DATA_DIR+test_file,  sep=r'\s+', header=None, names=col_names)
y_test = pd.read_csv(DATA_DIR+rul_file,   sep=r'\s+', header=None, names=['RemainingUsefulLife'])
print(train.shape)

(24720, 26)


In [3]:
def add_remaining_useful_life(df):
    g = df.groupby('unit_nr')
    mc = g['time_cycles'].max()
    rf = df.merge(mc.to_frame(name='max_cycle'), left_on='unit_nr', right_index=True)
    rf['RUL'] = rf['max_cycle'] - rf['time_cycles']
    return rf.drop('max_cycle', axis=1)

train = add_remaining_useful_life(train)
train['RUL'].clip(upper=125, inplace=True)

remaining_sensors = ['s_2','s_3','s_4','s_7','s_8','s_9','s_11','s_12','s_13',
                     's_14','s_15','s_17','s_20','s_21']
drop_sensors = [s for s in sensor_names if s not in remaining_sensors]

In [4]:
def add_operating_condition(df):
    d = df.copy()
    d['setting_1'] = d['setting_1'].round()
    d['setting_2'] = d['setting_2'].round(decimals=2)
    d['op_cond'] = (d['setting_1'].astype(str)+'_'+d['setting_2'].astype(str)+'_'+
                    d['setting_3'].astype(str))
    return d

def condition_scaler(df_tr, df_te, sensors):
    sc = StandardScaler()
    for c in df_tr['op_cond'].unique():
        sc.fit(df_tr.loc[df_tr['op_cond']==c, sensors])
        df_tr.loc[df_tr['op_cond']==c, sensors] = sc.transform(df_tr.loc[df_tr['op_cond']==c, sensors])
        df_te.loc[df_te['op_cond']==c, sensors] = sc.transform(df_te.loc[df_te['op_cond']==c, sensors])
    return df_tr, df_te

def exponential_smoothing(df, sensors, n, alpha=0.4):
    df = df.copy()
    df[sensors] = (df.groupby('unit_nr')[sensors]
                     .apply(lambda x: x.ewm(alpha=alpha).mean())
                     .reset_index(level=0, drop=True))
    def mask(a, s):
        r = np.ones_like(a); r[0:s] = 0; return r
    m = df.groupby('unit_nr')['unit_nr'].transform(mask, s=n).astype(bool)
    return df[m]

In [5]:
def gen_train_data(df, seq_len, cols):
    data = df[cols].values
    for start, stop in zip(range(0, len(data)-(seq_len-1)), range(seq_len, len(data)+1)):
        yield data[start:stop, :]

def gen_data_wrapper(df, seq_len, cols, units=np.array([])):
    if units.size <= 0: units = df['unit_nr'].unique()
    g = (list(gen_train_data(df[df['unit_nr']==u], seq_len, cols)) for u in units)
    return np.concatenate(list(g)).astype(np.float32)

def gen_labels(df, seq_len, label):
    m = df[label].values
    return m[seq_len-1:len(m), :]

def gen_label_wrapper(df, seq_len, label, units=np.array([])):
    if units.size <= 0: units = df['unit_nr'].unique()
    g = [gen_labels(df[df['unit_nr']==u], seq_len, label) for u in units]
    return np.concatenate(g).astype(np.float32)

def gen_test_data(df, seq_len, cols, mask_value):
    if df.shape[0] < seq_len:
        m = np.full((seq_len, len(cols)), mask_value)
        m[seq_len-df.shape[0]:, :] = df[cols].values
    else:
        m = df[cols].values
    yield m[m.shape[0]-seq_len:m.shape[0], :]

In [6]:
SEQUENCE_LENGTH = 20
X_tr = add_operating_condition(train.drop(drop_sensors, axis=1))
X_te = add_operating_condition(test.drop(drop_sensors, axis=1))
X_tr, X_te = condition_scaler(X_tr, X_te, remaining_sensors)
X_tr = exponential_smoothing(X_tr, remaining_sensors, 0, 0.4)
X_te = exponential_smoothing(X_te, remaining_sensors, 0, 0.4)

gss = GroupShuffleSplit(n_splits=1, train_size=0.80, random_state=SEED)
for tr_u, va_u in gss.split(X_tr['unit_nr'].unique(), groups=X_tr['unit_nr'].unique()):
    tr_u = X_tr['unit_nr'].unique()[tr_u]
    va_u = X_tr['unit_nr'].unique()[va_u]
train_split_array = gen_data_wrapper(X_tr, SEQUENCE_LENGTH, remaining_sensors, tr_u)
train_split_label = gen_label_wrapper(X_tr, SEQUENCE_LENGTH, ['RUL'], tr_u)
val_split_array   = gen_data_wrapper(X_tr, SEQUENCE_LENGTH, remaining_sensors, va_u)
val_split_label   = gen_label_wrapper(X_tr, SEQUENCE_LENGTH, ['RUL'], va_u)

test_gen = (list(gen_test_data(X_te[X_te['unit_nr']==u], SEQUENCE_LENGTH, remaining_sensors, -99.))
            for u in X_te['unit_nr'].unique())
test_array = np.concatenate(list(test_gen)).astype(np.float32)
N_FEATURES = train_split_array.shape[2]
print('train', train_split_array.shape, 'val', val_split_array.shape, 'test', test_array.shape)

train (18492, 20, 14) val (4328, 20, 14) test (100, 20, 14)


In [7]:
def build_sequential(n_conv=2, filters=128, kernel=3, n_lstm=2, lstm_units=128,
                     dense_units=200, dropout=0.2, lr=1e-3, optimizer='adam'):
    """Stacked sequential CNN-LSTM (matches the FD003 sequential baseline family)."""
    m = Sequential()
    m.add(Masking(mask_value=-99., input_shape=(SEQUENCE_LENGTH, N_FEATURES)))
    for _ in range(n_conv):
        m.add(Conv1D(filters, kernel_size=kernel, padding='same', activation='relu'))
        m.add(Dropout(0.0))
        m.add(MaxPooling1D(pool_size=2, padding='same'))
    m.add(TimeDistributed(Flatten()))
    for i in range(n_lstm):
        m.add(LSTM(lstm_units, return_sequences=(i < n_lstm-1),
                   activation='tanh' if i == n_lstm-1 else 'tanh'))
    m.add(Dense(dense_units, activation='relu'))
    m.add(Dropout(dropout))
    m.add(Dense(1, activation='linear'))
    opt = Adam(lr) if optimizer == 'adam' else RMSprop(lr)
    m.compile(loss='mean_squared_error', optimizer=opt)
    return m

def rmse(y, p):
    return float(np.sqrt(mean_squared_error(y, p)))

## Genetic Algorithm (DEAP)

Same GA configuration used for the parallel model — population 10, 5 generations,
two-point crossover (0.5), Gaussian mutation, tournament selection — here applied to the
sequential architecture. Fitness = validation RMSE.

In [8]:
filters_choices = [32, 64, 96, 128]
kernel_choices  = [3, 5]
lstm_choices    = [32, 64, 96, 128]
dense_choices   = [64, 128, 256]
dropout_choices = [0.2, 0.3, 0.4, 0.5]
lr_choices      = list(np.round(np.linspace(1e-4, 1e-2, 10), 6))

if 'FitnessMin' not in dir(creator):
    creator.create('FitnessMin', base.Fitness, weights=(-1.0,))
    creator.create('Individual', list, fitness=creator.FitnessMin)

tb = base.Toolbox()
tb.register('f',  np.random.randint, 0, len(filters_choices))
tb.register('k',  np.random.randint, 0, len(kernel_choices))
tb.register('nc', np.random.randint, 1, 3)     # 1-2 conv blocks
tb.register('lu', np.random.randint, 0, len(lstm_choices))
tb.register('nl', np.random.randint, 1, 3)     # 1-2 lstm layers
tb.register('du', np.random.randint, 0, len(dense_choices))
tb.register('dr', np.random.randint, 0, len(dropout_choices))
tb.register('lr', np.random.randint, 0, len(lr_choices))
tb.register('individual', tools.initCycle, creator.Individual,
            (tb.f, tb.k, tb.nc, tb.lu, tb.nl, tb.du, tb.dr, tb.lr), n=1)
tb.register('population', tools.initRepeat, list, tb.individual)

def evaluate(ind):
    try:
        m = build_sequential(n_conv=ind[2], filters=filters_choices[ind[0]],
                             kernel=kernel_choices[ind[1]], n_lstm=ind[4],
                             lstm_units=lstm_choices[ind[3]], dense_units=dense_choices[ind[5]],
                             dropout=dropout_choices[ind[6]], lr=lr_choices[ind[7]])
        es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        m.fit(train_split_array, train_split_label,
              validation_data=(val_split_array, val_split_label),
              epochs=20, batch_size=32, callbacks=[es], verbose=0)
        return (rmse(val_split_label, m.predict(val_split_array, verbose=0)),)
    except Exception as e:
        print('eval failed:', e); return (1e5,)

tb.register('evaluate', evaluate)
tb.register('mate', tools.cxTwoPoint)
tb.register('mutate', tools.mutGaussian, mu=0, sigma=1, indpb=0.2)
tb.register('select', tools.selTournament, tournsize=3)

In [9]:
POP, NGEN = 10, 5
pop = tb.population(n=POP)
for gen in range(NGEN):
    print('Generation', gen+1)
    off = algorithms.varAnd(pop, tb, cxpb=0.5, mutpb=0.3)
    for ind in off:                       # keep integer genes valid after mutation
        ind[0]=int(np.clip(round(ind[0]),0,len(filters_choices)-1))
        ind[1]=int(np.clip(round(ind[1]),0,len(kernel_choices)-1))
        ind[2]=int(np.clip(round(ind[2]),1,2))
        ind[3]=int(np.clip(round(ind[3]),0,len(lstm_choices)-1))
        ind[4]=int(np.clip(round(ind[4]),1,2))
        ind[5]=int(np.clip(round(ind[5]),0,len(dense_choices)-1))
        ind[6]=int(np.clip(round(ind[6]),0,len(dropout_choices)-1))
        ind[7]=int(np.clip(round(ind[7]),0,len(lr_choices)-1))
    for ind, fit in zip(off, map(tb.evaluate, off)):
        ind.fitness.values = fit
    pop = tb.select(off, k=len(pop))

best = tools.selBest(pop, 1)[0]
print('Best genes:', list(best), 'val RMSE:', best.fitness.values[0])

Generation 1
Generation 2
Generation 3
Generation 4
Generation 5
Best genes: [2, 0, 2, 2, 2, 0, 3, 0] val RMSE: 13.243487358093262


In [10]:
# retrain best GA configuration and evaluate on the FD003 test set
ga_model = build_sequential(n_conv=best[2], filters=filters_choices[best[0]],
                            kernel=kernel_choices[best[1]], n_lstm=best[4],
                            lstm_units=lstm_choices[best[3]], dense_units=dense_choices[best[5]],
                            dropout=dropout_choices[best[6]], lr=lr_choices[best[7]])
ga_model.fit(train_split_array, train_split_label,
             validation_data=(val_split_array, val_split_label),
             epochs=100, batch_size=32,
             callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
             verbose=1)
ga_rmse = rmse(y_test.values, ga_model.predict(test_array, verbose=0))
print('GA sequential test RMSE:', round(ga_rmse, 4))

Epoch 1/100
578/578 [==============================] - 14s 18ms/step - loss: 5961.2920 - val_loss: 3025.9407
Epoch 2/100
578/578 [==============================] - 11s 20ms/step - loss: 2414.5276 - val_loss: 1757.4275
Epoch 3/100
578/578 [==============================] - 11s 19ms/step - loss: 1447.6902 - val_loss: 460.7448
Epoch 4/100
578/578 [==============================] - 11s 20ms/step - loss: 698.8188 - val_loss: 332.6111
Epoch 5/100
578/578 [==============================] - 11s 18ms/step - loss: 636.9097 - val_loss: 282.7519
Epoch 6/100
578/578 [==============================] - 9s 16ms/step - loss: 585.7768 - val_loss: 291.1291
Epoch 7/100
578/578 [==============================] - 11s 20ms/step - loss: 575.1526 - val_loss: 254.2204
Epoch 8/100
578/578 [==============================] - 10s 18ms/step - loss: 549.0064 - val_loss: 225.9929
Epoch 9/100
578/578 [==============================] - 10s 18ms/step - loss: 530.6309 - val_loss: 231.4298
Epoch 10/100
578/578 [===========

## Hyperband (Keras Tuner)

Same search scope Hyperband was given for the parallel model — the training/head
hyperparameters only (optimizer, learning rate, dropout, dense units) with the
convolutional and recurrent structure fixed — factor 3, max_epochs 30.

In [11]:
def hb_build(hp):
    opt = hp.Choice('optimizer', ['adam', 'rmsprop'])
    lr  = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
    dr  = hp.Float('dropout', 0.1, 0.5, step=0.1)
    du  = hp.Choice('dense_units', [64, 128, 256])
    return build_sequential(n_conv=2, filters=128, kernel=3, n_lstm=2, lstm_units=128,
                            dense_units=du, dropout=dr, lr=lr, optimizer=opt)

tuner = kt.Hyperband(hb_build, objective='val_loss', max_epochs=30, factor=3,
                     directory='hb_cmapss_seq', project_name='seq_optionC', overwrite=True)
tuner.search(train_split_array, train_split_label,
             validation_data=(val_split_array, val_split_label),
             epochs=30, batch_size=32,
             callbacks=[EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
                        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5)])
hb_model = tuner.get_best_models(1)[0]
hb_model.fit(train_split_array, train_split_label,
             validation_data=(val_split_array, val_split_label),
             epochs=50, batch_size=32,
             callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
             verbose=1)
hb_rmse = rmse(y_test.values, hb_model.predict(test_array, verbose=0))
print('Hyperband sequential test RMSE:', round(hb_rmse, 4))

Trial 90 Complete [00h 06m 16s]
val_loss: 288.09942626953125

Best val_loss So Far: 169.47543334960938
Total elapsed time: 01h 50m 11s
Epoch 1/50
578/578 [==============================] - 12s 16ms/step - loss: 172.2021 - val_loss: 178.3102
Epoch 2/50
578/578 [==============================] - 9s 15ms/step - loss: 165.8749 - val_loss: 175.3880
Epoch 3/50
578/578 [==============================] - 9s 15ms/step - loss: 164.7356 - val_loss: 203.2742
Epoch 4/50
578/578 [==============================] - 9s 15ms/step - loss: 159.4411 - val_loss: 188.5246
Epoch 5/50
578/578 [==============================] - 9s 15ms/step - loss: 153.1439 - val_loss: 187.8873
Epoch 6/50
578/578 [==============================] - 9s 15ms/step - loss: 152.1010 - val_loss: 199.9720
Epoch 7/50
578/578 [==============================] - 9s 15ms/step - loss: 152.1491 - val_loss: 172.3095
Epoch 8/50
578/578 [==============================] - 9s 15ms/step - loss: 145.6869 - val_loss: 203.2178
Epoch 9/50
578/578 [====

In [12]:
import json
summary = {'dataset':'C-MAPSS FD003','architecture':'sequential','window':SEQUENCE_LENGTH,
           'baseline_test_rmse':15.31,
           'ga_test_rmse':round(ga_rmse,4),
           'hyperband_test_rmse':round(hb_rmse,4)}
print(summary)
os.makedirs('output', exist_ok=True)
json.dump(summary, open('output/cmapss_fd003_sequential_optionC.json','w'), indent=2)

{'dataset': 'C-MAPSS FD003', 'architecture': 'sequential', 'window': 20, 'baseline_test_rmse': 15.31, 'ga_test_rmse': 15.8444, 'hyperband_test_rmse': 16.2572}
